# 01 — Data Loading
**Where Should I Buy? · GeoAI Course**

This notebook demonstrates loading all four data layers and inspecting them.

Goals:
- Load transactions, neighborhoods, transit stops, and parks
- Validate CRS (must be EPSG:4326)
- Inspect geometry types and column schemas
- Plot a quick overview map

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import geopandas as gpd
import matplotlib.pyplot as plt
from loguru import logger

from src.data.loaders import (
    load_transactions,
    load_neighborhoods,
    load_transit_stops,
    load_parks,
)
from src.geo.crs_utils import crs_summary, assert_wgs84
from src.data.validation import (
    validate_transactions,
    validate_neighborhoods,
    validate_transit_stops,
    validate_parks,
)

print('✅ Imports OK')

## 1. Load All Data Layers

In [ ]:
transactions  = load_transactions()
neighborhoods = load_neighborhoods()
transit_stops = load_transit_stops()
parks         = load_parks()

print('\n--- Data Summary ---')
print(crs_summary(transactions,  'Transactions'))
print(crs_summary(neighborhoods, 'Neighborhoods'))
print(crs_summary(transit_stops, 'Transit Stops'))
print(crs_summary(parks,         'Parks'))

## 2. Validate All Layers

In [ ]:
r1 = validate_transactions(transactions)
r2 = validate_neighborhoods(neighborhoods)
r3 = validate_transit_stops(transit_stops)
r4 = validate_parks(parks)

all_passed = all([r1.passed, r2.passed, r3.passed, r4.passed])
print(f'\n✅ All validations passed: {all_passed}')

## 3. Inspect Transactions

In [ ]:
print(f'Shape: {transactions.shape}')
print(f'Columns: {list(transactions.columns)}')
transactions.head(5)

In [ ]:
# Price distribution by neighborhood
transactions.groupby('neighborhood')['transaction_price'].describe().round(0)

## 4. Quick Overview Map

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot layers
neighborhoods.plot(ax=ax, color='lightblue', edgecolor='navy', alpha=0.5, label='Neighborhoods')
transit_stops.plot(ax=ax, color='orange', markersize=3, alpha=0.6, label='Transit Stops')
parks.plot(ax=ax, color='green', markersize=5, alpha=0.7, label='Parks')
transactions.plot(ax=ax, color='red', markersize=2, alpha=0.3, label='Transactions')

ax.set_title('Gush Dan — All Data Layers', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('../data/processed/overview_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Map saved to data/processed/overview_map.png')